# z-Only CSE Control with z-Only Observations

This notebook performs the first direct information-channel test proposed by Takemasa:

- **observation:** $z$ only;
- **intervention:** $u_z$ only;
- **control objective:** retain the sign of $x$;
- **CSE decision rule:** unchanged.

The experiment uses the independently generated z-only OSSE analysis history from notebook 14. The CSE core in `class_cse_qiwen.py` is not modified. A thin adapter passes only the generated $z$ observation to the one-channel DA update while preserving the original three random draws, so the observation-noise stream remains paired with the existing full-observation experiment.

As in the validated nature-reference configuration, the fallback branch retains one additional binary piece of objective information: the **current sign of the controlled $x$ state** is used to select a past safe trajectory. This is not a full $x$ observation in the DA update, and no future uncontrolled nature state is used. Stating this boundary explicitly is essential: this notebook tests z-only *assimilation* while retaining the sign information that Takemasa identified as necessary for judging the control objective.


In [ ]:
from pathlib import Path
import importlib
import pickle

import matplotlib.pyplot as plt
import numpy as np

from class_da_sys import da_system
import class_cse_qiwen as cse_qiwen
import class_cse_observation_experiment as observation_experiment
from project_paths import ANALYSIS_DATA_DIR, CSE_ROOT, OSSE_DATA_DIR

cse_qiwen = importlib.reload(cse_qiwen)
observation_experiment = importlib.reload(observation_experiment)

OBSERVATION_LABEL = 'z_only_obs'
EXPERIMENT_NAME = 'z_control_z_obs_nature_qiwenevents'
OBSERVED_INDICES = (2,)
CONTROL_INDICES = [2]

ANALYSIS_FILE = (
    ANALYSIS_DATA_DIR.parent / OBSERVATION_LABEL / 'data'
    / 'x_analysis_init_z_only_obs.pkl'
)
OSSE_EXPERIMENT_ROOT = OSSE_DATA_DIR.parent / OBSERVATION_LABEL
OSSE_EXPERIMENT_DATA_DIR = OSSE_EXPERIMENT_ROOT / 'data'
EXPERIMENT_ROOT = CSE_ROOT / EXPERIMENT_NAME
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
FIGURES_DIR = EXPERIMENT_ROOT / 'figures'
for directory in (RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print('DA-system input:', ANALYSIS_FILE)
print('OSSE input:', OSSE_EXPERIMENT_DATA_DIR)
print('CSE output:', EXPERIMENT_ROOT)


## Load the z-Only OSSE and Validate the Information Boundary


In [ ]:
das = da_system().load(ANALYSIS_FILE)
das.multiplicative_inflation = 1.04
x_nature = np.asarray(das.getStateVector().getTrajectory(), dtype=float)

with open(OSSE_EXPERIMENT_DATA_DIR / 'osse_Xa_history.pkl', 'rb') as stream:
    osse_Xa_history = np.asarray(pickle.load(stream), dtype=float)
with open(OSSE_EXPERIMENT_DATA_DIR / 'osse_summary.pkl', 'rb') as stream:
    osse_summary = pickle.load(stream)

assert das.xdim == 3 and das.ydim == 1
assert np.array_equal(np.asarray(das.H), np.array([[0.0, 0.0, 1.0]]))
assert np.asarray(das.R).shape == (1, 1)
assert osse_summary['observed_indices'] == [2]
assert osse_Xa_history.shape[1:] == (das.xdim, das.edim)

Ta = das.acyc_step
fcst_dt = das.fcst_dt
multiple_number = 4
T_0 = 75.1
position = int(np.ceil(T_0 * multiple_number))

print('H =', np.asarray(das.H))
print('Observed indices:', OBSERVED_INDICES)
print('Controlled indices:', CONTROL_INDICES)
print("Fallback regime-sign source: current controlled x (regime_reference='nature')")
print('OSSE analysis history:', osse_Xa_history.shape)


## Select the Same Qiwen Event Family

Nature-run crossings define a reproducible evaluation set and aligned initial times. They do not enter the cycle-by-cycle intervention rule. Once the experiment begins, DA receives only $z$. The existing fallback rule may additionally read the current sign of controlled $x$ because `regime_reference='nature'`; it never reads the future uncontrolled transition.


In [ ]:
regime_change_info = cse_qiwen.detect_regime_changes(
    x_nature,
    RESULTS_DIR,
    position=position,
    output_filename='change_region_time_z_obs_qiwenevents.pkl',
    event_selection='qiwen',
)
t_start_series = regime_change_info['t_start_series']

D = 0.05
experiment_step = 1000
EVENT_INDEX = 1

if EVENT_INDEX >= len(t_start_series):
    raise IndexError(f'EVENT_INDEX={EVENT_INDEX} but only {len(t_start_series)} events are available.')

t_start = int(t_start_series[EVENT_INDEX])
t0 = (t_start - position) - (t_start - position) % Ta
osse_start_index = (t0 - osse_summary['t0']) // Ta
if osse_summary['t0'] + osse_start_index * Ta != t0:
    raise ValueError('t0 is not aligned with the z-only OSSE grid.')

print('Event index:', EVENT_INDEX)
print('Crossing used to define the test window:', t_start)
print('Aligned CSE start:', t0)
print('z-only OSSE start index:', osse_start_index)


## Run One z-Observation / z-Control Event


In [ ]:
single_cse_result = observation_experiment.run_single_cse_with_observation_subset(
    das=das,
    observed_indices=OBSERVED_INDICES,
    run_single_cse_core=cse_qiwen.run_single_cse,
    event_index=EVENT_INDEX,
    D_run=D,
    seed=1,
    x_nature=x_nature,
    t_start_series=t_start_series,
    osse_Xa_history=osse_Xa_history,
    osse_summary=osse_summary,
    experiment_step=experiment_step,
    position=position,
    Ta=Ta,
    fcst_dt=fcst_dt,
    path_figures=FIGURES_DIR,
    total_events=len(t_start_series),
    regime_reference='nature',
    candidate_pool='full',
    control_indices=CONTROL_INDICES,
    T_0=T_0,
    multiple_number=multiple_number,
)

D_TAG = f'{D:g}'.replace('.', 'p')
FULL_RESULT_PATH = RESULTS_DIR / (
    f'cse_z_control_z_obs_event_{EVENT_INDEX}_D{D_TAG}_qiwenevents.pkl'
)
temporary_path = FULL_RESULT_PATH.with_suffix(FULL_RESULT_PATH.suffix + '.tmp')
with temporary_path.open('wb') as stream:
    pickle.dump(single_cse_result, stream, pickle.HIGHEST_PROTOCOL)
temporary_path.replace(FULL_RESULT_PATH)

COMPARISON_PATH = RESULTS_DIR / (
    f'cse_z_control_z_obs_nature_run_event_{EVENT_INDEX}_D{D_TAG}_qiwenevents.pkl'
)
cse_qiwen.export_nature_run_comparison(
    single_cse_result,
    x_nature,
    COMPARISON_PATH,
)

assert single_cse_result['yo_history'].shape[1] == 1
assert single_cse_result['yo_history_full_generated'].shape[1] == 3
assert np.allclose(
    single_cse_result['yo_history'][:, 0],
    single_cse_result['yo_history_full_generated'][:, 2],
)

print('Success:', single_cse_result['success'])
print('Failure cycle:', single_cse_result['failure_cycle'])
print('Completed cycles:', single_cse_result['n_completed'])
print('Control ratio:', single_cse_result['control_ratio'])
print('Branch counts:', single_cse_result['branch_counts'])
print('Assimilated observation shape:', single_cse_result['yo_history'].shape)
print('Full generated observation shape:', single_cse_result['yo_history_full_generated'].shape)
print('Full result:', FULL_RESULT_PATH)
print('Nature-run comparison:', COMPARISON_PATH)


## z-Only Analysis Ensemble and Lorenz Symmetry

The preceding trajectory figures show the controlled truth, not the DA ensemble. This diagnostic compares the analysis ensemble at the beginning of every CSE cycle with both the controlled state and its Lorenz-symmetric counterpart,

\[
(x,y,z) \longrightarrow (-x,-y,z).
\]

If observations of $z$ cannot identify the active lobe, the ensemble may remain close to the symmetry-related state even while the controlled truth stays in the desired regime.


In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np

from project_paths import CSE_ROOT

symmetry_experiment_root = CSE_ROOT / 'z_control_z_obs_nature_qiwenevents'
FIGURES_DIR = symmetry_experiment_root / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if 'single_cse_result' not in globals():
    event_for_plot = int(globals().get('EVENT_INDEX', 1))
    D_for_plot = float(globals().get('D', 0.05))
    D_tag_for_plot = f'{D_for_plot:g}'.replace('.', 'p')
    saved_result_path = symmetry_experiment_root / 'results' / (
        f'cse_z_control_z_obs_event_{event_for_plot}_D{D_tag_for_plot}_qiwenevents.pkl'
    )
    with saved_result_path.open('rb') as stream:
        single_cse_result = pickle.load(stream)

Ta = int(single_cse_result['cse_summary']['Ta'])
analysis_ensembles = np.asarray(
    single_cse_result['Xa_cycle_initial_history'],
    dtype=float,
)
n_analysis_cycles = analysis_ensembles.shape[0]
analysis_steps = np.arange(n_analysis_cycles) * Ta
controlled_at_analysis = np.asarray(
    single_cse_result['evo_history'],
    dtype=float,
)[:, analysis_steps].T
analysis_mean = analysis_ensembles.mean(axis=2)

symmetry_sign = np.array([-1.0, -1.0, 1.0])
symmetry_counterpart = controlled_at_analysis * symmetry_sign

rmse_to_controlled = np.sqrt(
    np.mean((analysis_mean - controlled_at_analysis) ** 2, axis=1)
)
rmse_to_symmetry = np.sqrt(
    np.mean((analysis_mean - symmetry_counterpart) ** 2, axis=1)
)
x_sign_agreement = np.mean(
    np.sign(analysis_ensembles[:, 0, :])
    == np.sign(controlled_at_analysis[:, 0, None])
)

fig, axes = plt.subplots(3, 1, figsize=(11, 8.2), sharex=True)
state_labels = ('x', 'y', 'z')

for state_index, ax in enumerate(axes):
    ax.plot(
        analysis_steps,
        controlled_at_analysis[:, state_index],
        color='tab:blue',
        linewidth=1.5,
        label='controlled state',
        zorder=4,
    )
    if state_index < 2:
        ax.plot(
            analysis_steps,
            symmetry_counterpart[:, state_index],
            color='0.35',
            linestyle=':',
            linewidth=1.3,
            label='symmetry counterpart',
            zorder=2,
        )
        ax.axhline(0.0, color='0.75', linewidth=0.8)
    for member in range(analysis_ensembles.shape[2]):
        ax.plot(
            analysis_steps,
            analysis_ensembles[:, state_index, member],
            color='lightcoral',
            linewidth=0.65,
            alpha=0.35,
            label='analysis members' if member == 0 else None,
            zorder=1,
        )
    ax.plot(
        analysis_steps,
        analysis_mean[:, state_index],
        color='darkred',
        linestyle='--',
        linewidth=1.2,
        label='analysis mean',
        zorder=3,
    )
    ax.set_ylabel(state_labels[state_index])
    ax.grid(alpha=0.2)

legend_handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(
    legend_handles,
    legend_labels,
    frameon=False,
    ncol=4,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.945),
)
axes[0].text(
    0.01,
    0.03,
    'x-sign agreement = {:.1f}%\nMean RMSE to controlled state = {:.2f}\nMean RMSE to symmetry counterpart = {:.2f}'.format(
        100.0 * x_sign_agreement,
        np.mean(rmse_to_controlled),
        np.mean(rmse_to_symmetry),
    ),
    transform=axes[0].transAxes,
    ha='left',
    va='bottom',
    fontsize=9,
    bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.9},
)
axes[-1].set_xlabel('Steps after CSE initialization')
fig.suptitle(
    'z-only DA follows the symmetry-related Lorenz state',
    y=0.995,
)
fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.90))

ensemble_symmetry_path = FIGURES_DIR / 'z_only_analysis_ensemble_symmetry_comparison.png'
fig.savefig(ensemble_symmetry_path, dpi=300, bbox_inches='tight')
plt.show()

print('x-sign agreement across all members and cycles:', x_sign_agreement)
print('Mean RMSE to controlled state:', float(np.mean(rmse_to_controlled)))
print('Mean RMSE to symmetry counterpart:', float(np.mean(rmse_to_symmetry)))
print('Figure:', ensemble_symmetry_path)


## Free versus Controlled Nature Trajectories


In [ ]:
n_completed = int(single_cse_result['n_completed'])
n_steps = n_completed * Ta
controlled = np.asarray(single_cse_result['evo_history'], dtype=float)[:, :n_steps + 1]
free = x_nature[:, single_cse_result['t0']:single_cse_result['t0'] + n_steps + 1]
relative_steps = np.arange(n_steps + 1)

interventions = np.zeros_like(controlled)
for cycle, item in enumerate(single_cse_result['intervention_history']):
    start = cycle * Ta
    interventions[:, start:start + Ta] = np.asarray(item['perturbations']).T

fig, axes = plt.subplots(4, 1, figsize=(11, 9.5), sharex=True)
labels = ('x', 'y', 'z')
for state_index, ax in enumerate(axes[:3]):
    ax.plot(relative_steps, free[state_index], color='0.35', linewidth=1.15, label='free nature run')
    ax.plot(relative_steps, controlled[state_index], color='tab:blue', linestyle='--', linewidth=1.35, label='controlled nature run')
    if state_index < 2:
        ax.axhline(0.0, color='0.75', linewidth=0.8)
    ax.set_ylabel(labels[state_index])
    ax.grid(alpha=0.2)

nonzero = np.flatnonzero(np.abs(interventions[2]) > 0)
axes[3].vlines(nonzero, 0.0, interventions[2, nonzero], color='tab:red', linewidth=0.8)
axes[3].axhline(0.0, color='0.35', linewidth=0.8)
axes[3].set_ylabel(r'$u_z$')
axes[3].set_xlabel('Steps after CSE initialization')
axes[3].grid(alpha=0.2)
legend_handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(
    legend_handles,
    legend_labels,
    frameon=False,
    ncol=2,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.945),
)
fig.suptitle(
    'z-only observations and z-only interventions: single-event outcome',
    y=0.995,
)
fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.90))
comparison_figure_path = FIGURES_DIR / 'z_obs_z_control_free_vs_controlled.png'
fig.savefig(comparison_figure_path, dpi=300, bbox_inches='tight')
plt.show()

print('Figure:', comparison_figure_path)


## Three-Dimensional Attractor Comparison


In [ ]:
plot_3d_data = cse_qiwen.plot_3d_nature_run_comparison_with_fallback(
    current_result=single_cse_result,
    x_nature=x_nature,
    Ta=Ta,
    path_figures=FIGURES_DIR,
    analysis_data_path=ANALYSIS_FILE,
    results_dir=RESULTS_DIR,
    output_tag='_z_obs_qiwenevents',
    D=D,
    event_index=EVENT_INDEX,
    experiment_step=experiment_step,
    control_label='z_obs_z_control',
    output_filename='NR_3D_comparison_z_obs_z_control.png',
)

print('3D figure:', plot_3d_data['output_path'])


## Three-Dimensional Free Trajectory versus z-Only Analysis Ensemble

This companion figure replaces the controlled trajectory in the right-hand panel with the analysis ensemble at every DA-cycle initialization. It directly tests the Lorenz symmetry ambiguity: the free trajectory visits both lobes, whereas the z-only analyses are expected to populate the symmetry-related negative lobe.


In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np

from class_da_sys import da_system
from project_paths import ANALYSIS_DATA_DIR, CSE_ROOT

analysis_3d_experiment_root = CSE_ROOT / 'z_control_z_obs_nature_qiwenevents'
analysis_3d_figures_dir = analysis_3d_experiment_root / 'figures'
analysis_3d_figures_dir.mkdir(parents=True, exist_ok=True)

if 'single_cse_result' not in globals():
    event_for_plot = int(globals().get('EVENT_INDEX', 1))
    D_for_plot = float(globals().get('D', 0.05))
    D_tag_for_plot = f'{D_for_plot:g}'.replace('.', 'p')
    saved_result_path = analysis_3d_experiment_root / 'results' / (
        f'cse_z_control_z_obs_event_{event_for_plot}_D{D_tag_for_plot}_qiwenevents.pkl'
    )
    with saved_result_path.open('rb') as stream:
        single_cse_result = pickle.load(stream)

if 'x_nature' not in globals():
    z_only_analysis_file = (
        ANALYSIS_DATA_DIR.parent / 'z_only_obs' / 'data'
        / 'x_analysis_init_z_only_obs.pkl'
    )
    das_3d = da_system().load(z_only_analysis_file)
    x_nature = np.asarray(das_3d.getStateVector().getTrajectory(), dtype=float)

Ta = int(single_cse_result['cse_summary']['Ta'])
n_completed_3d = int(single_cse_result['n_completed'])
n_steps_3d = n_completed_3d * Ta
t0_3d = int(single_cse_result['t0'])
free_trajectory_3d = np.asarray(
    x_nature[:, t0_3d:t0_3d + n_steps_3d + 1],
    dtype=float,
)
analysis_ensembles_3d = np.asarray(
    single_cse_result['Xa_cycle_initial_history'],
    dtype=float,
)
analysis_mean_3d = analysis_ensembles_3d.mean(axis=2)

fig = plt.figure(figsize=(14, 7.2), facecolor='white')
ax_free = fig.add_subplot(121, projection='3d')
ax_analysis = fig.add_subplot(122, projection='3d')

free_artist = ax_free.scatter(
    free_trajectory_3d[0],
    free_trajectory_3d[1],
    free_trajectory_3d[2],
    c=np.linspace(0.0, 1.0, free_trajectory_3d.shape[1]),
    cmap='turbo',
    s=3,
    alpha=0.8,
    label='free nature trajectory',
)

for member in range(analysis_ensembles_3d.shape[2]):
    analysis_artist = ax_analysis.scatter(
        analysis_ensembles_3d[:, 0, member],
        analysis_ensembles_3d[:, 1, member],
        analysis_ensembles_3d[:, 2, member],
        color='lightcoral',
        s=5,
        alpha=0.35,
        label='analysis members' if member == 0 else None,
    )

mean_artist, = ax_analysis.plot(
    analysis_mean_3d[:, 0],
    analysis_mean_3d[:, 1],
    analysis_mean_3d[:, 2],
    color='darkred',
    linewidth=1.0,
    alpha=0.9,
    label='analysis mean',
)

for ax in (ax_free, ax_analysis):
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    ax.set_xlim(-25, 25)
    ax.set_ylim(-20, 20)
    ax.set_zlim(0, 45)
    ax.grid(alpha=0.25)
    ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
    ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))
    ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0))

ax_free.set_title('Free nature trajectory', pad=8)
ax_analysis.set_title('z-only analysis ensemble', pad=8)
ax_analysis.text2D(
    0.03,
    0.04,
    'Analysis x range: [{:.2f}, {:.2f}]'.format(
        np.min(analysis_ensembles_3d[:, 0, :]),
        np.max(analysis_ensembles_3d[:, 0, :]),
    ),
    transform=ax_analysis.transAxes,
    fontsize=9,
    bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.9},
)

fig.legend(
    [free_artist, analysis_artist, mean_artist],
    ['free nature trajectory', 'analysis members', 'analysis mean'],
    frameon=False,
    ncol=3,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.93),
)
fig.suptitle('Free trajectory versus z-only analysis ensemble', y=0.995)
fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.88))

analysis_3d_path = (
    analysis_3d_figures_dir
    / 'free_nature_vs_z_only_analysis_ensemble_3d.png'
)
fig.savefig(analysis_3d_path, dpi=300, bbox_inches='tight')
plt.show()

print('Analysis x range:', float(np.min(analysis_ensembles_3d[:, 0, :])),
      float(np.max(analysis_ensembles_3d[:, 0, :])))
print('3D analysis-ensemble figure:', analysis_3d_path)


## Interpretation Boundary

This single event answers whether the current CSE procedure can still achieve the desired trajectory when its **analysis updates** receive no direct information from $x$ or $y$, while its validated fallback retains the current sign of controlled $x$. It does not by itself establish a success rate. A later batch notebook should repeat the same paired comparison over the full event set; that notebook is deliberately deferred until this pilot has been run and checked.
